In [1]:
import dlt
import pandas as pd

@dlt.resource
def taxi_data():
    url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-01.parquet"
    df = pd.read_parquet(url)
    yield df

pipeline = dlt.pipeline(
    pipeline_name="taxi_pipeline",
    destination="duckdb",
    dataset_name="taxi_data"
)

load_info = pipeline.run(taxi_data())
print(load_info)

2026-02-22 09:27:56,700|[WARNING]|19185|137220925413184|dlt|validate.py|verify_normalized_table:91|In schema `taxi`: The following columns in table 'taxi_data' did not receive any data during this load and therefore could not have their types inferred:
  - ehail_fee

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'ehail_fee': {'data_type': 'text'}})



Pipeline taxi_pipeline load step completed in 0.40 seconds
1 load package(s) were loaded to destination duckdb and into dataset taxi_data
The duckdb destination used duckdb:////workspaces/dataclub-dataengineering/taxi_pipeline.duckdb location to store data
Load package 1771752476.5137115 is LOADED and contains no failed jobs


In [2]:
with pipeline.sql_client() as client:
    with client.execute_query("SELECT COUNT(*) FROM taxi_data") as cursor:
        print(cursor.df())

   count_star()
0        136422


In [3]:
@dlt.resource
def taxi_data():
    for month in range(1, 3):  # start small for test
        month_str = str(month).zfill(2)
        url = f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-{month_str}.parquet"
        yield url

In [4]:
import dlt

@dlt.resource
def taxi_data():
    for month in range(1, 4):  # start with 3 months
        month_str = str(month).zfill(2)
        yield f"https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2023-{month_str}.parquet"

In [6]:
pipeline = dlt.pipeline(
    pipeline_name="taxi_pipeline",
    destination="duckdb",
    dataset_name="taxi_data"
)

load_info = pipeline.run(taxi_data())
print(load_info)

2026-02-22 09:30:59,506|[WARNING]|19185|137220925413184|dlt|pipeline.py|run:734|The pipeline `run` method will now load the pending load packages. The data you passed to the run function will not be extracted. In order to do that you must run the pipeline again
2026-02-22 09:30:59,551|[WARNING]|19185|137220925413184|dlt|job_client_impl.py|_check_table_update_hints:782|Column(s) ['"_dlt_id"'] with hint unique are being added to existing table taxi_data. Several hint types may not be added to existing tables.
2026-02-22 09:30:59,551|[WARNING]|19185|137220925413184|dlt|job_client_impl.py|_check_table_update_hints:776|Column(s) ['"_dlt_load_id"', '"_dlt_id"'] with NOT NULL are being added to existing table taxi_data. If there's data in the table the operation will fail.


PipelineStepFailed: Pipeline execution failed at `step=load` when processing package with `load_id=1771752632.4684582` with exception:

<class 'dlt.destinations.exceptions.DatabaseTransientException'>
Parser Error: Adding columns with constraints not yet supported

Pending packages are left in the pipeline and will be re-tried on the next pipeline run. If you pass new data to extract to next run, it will be ignored. Run `dlt pipeline taxi_pipeline info` for more information or `dlt pipeline taxi_pipeline drop-pending-packages` to drop pending packages.

In [ ]:
import os
os.environ["DESTINATION__CREDENTIALS"] = os.environ["GCP_CREDENTIALS"]

In [ ]:
os.environ["BUCKET_URL"] = "gs://dataclub-s"

In [1]:
pip install dlt[bigquery,gs]
pip install dlt[duckdb]
pip install pandas requests pyarrow
pip install google-cloud-storage

SyntaxError: invalid syntax (2527434312.py, line 1)

In [2]:
python --version


NameError: name 'python' is not defined

In [1]:
# Install everything needed for Module 4
!pip install --upgrade pip
!pip install dlt[bigquery,gs] dlt[duckdb] pandas requests pyarrow google-cloud-storage duckdb

In [2]:
import os

# Path to your service account JSON
os.environ["DESTINATION__CREDENTIALS"] = "/workspaces/dataclub-dataengineering/data-system/gcp_service_account.json"

# Your GCS bucket
os.environ["BUCKET_URL"] = "gs://dataclub-s"

In [3]:
import dlt
import requests
import pandas as pd
from io import BytesIO
from dlt.destinations import filesystem

In [5]:
@dlt.resource(name="rides", write_disposition="replace")
def download_parquet():
    prefix = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata"
    for month in range(1, 7):
        url = f"{prefix}_2024-0{month}.parquet"
        print(f"Downloading {url}")
        response = requests.get(url)
        df = pd.read_parquet(BytesIO(response.content))
        yield df

In [6]:
pipeline = dlt.pipeline(
    pipeline_name="rides_pipeline",
    destination="duckdb",  # Change to "bigquery" if needed
    dataset_name="rides_dataset"
)

In [7]:
load_info = pipeline.run(download_parquet())
print(load_info)

Pipeline rides_pipeline load step completed in 23.70 seconds
1 load package(s) were loaded to destination duckdb and into dataset rides_dataset
The duckdb destination used duckdb:////workspaces/dataclub-dataengineering/data-system/gcp_service_account.json location to store data
Load package 1771765415.523104 is LOADED and contains no failed jobs


In [8]:
from google.cloud import bigquery

bq_client = bigquery.Client()
dataset_id = f"{bq_client.project}.trips_data_all"

dataset = bigquery.Dataset(dataset_id)
dataset.location = "US"

# Create dataset if not exists
dataset = bq_client.create_dataset(dataset, exists_ok=True)
print(f"Dataset {dataset_id} ready")

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [9]:
import os
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/workspaces/dataclub-dataengineering/data-system/gcp_service_account.json"

In [ ]:
from google.cloud import bigquery

bq_client = bigquery.Client()
print("Client ready, project:", bq_client.project)

In [2]:
from google.oauth2 import service_account

credentials = service_account.Credentials.from_service_account_file(
    "/workspaces/dataclub-dataengineering/data-system/gcp_service_account.json"
)

bq_client = bigquery.Client(credentials=credentials, project=credentials.project_id)
print("Client ready, project:", bq_client.project)

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x94 in position 0: invalid start byte